In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), ".."))


# Original analyzer for data loading
from src.analysis.summary import EEGSummarizedAnalyzer
from src.definitions.fields import (
    ExperimentNames,
    CoordinateSystems,
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
    SingleDataMetadata,
)
from src.definitions.constants import ProjectPaths

# Modular analysis pipeline
from src.analysis.isc import (
    compute_mean_variance,
    compute_sliding_window_mean_variance,
)
from src.visualization.isc_plots import (
    plot_mean_variance_distribution,
    plot_sliding_window_mean_variance,
    plot_band_mean_variance_distributions,
    plot_band_sliding_window_mean_variance,
    print_data_overview,
)

from xvfbwrapper import Xvfb

vdisplay = Xvfb()
vdisplay.start()

%matplotlib inline

In [ ]:
# Directory where all figures from this notebook are saved
SAVE_DIR = ProjectPaths.PLOTS_PATH / "MeanVarianceAnalysis"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {SAVE_DIR}")

In [ ]:
# Sliding-window parameters
WINDOW_SEC = 5.0
STEP_SEC = 2.5

## Setup

Create `EEGSummarizedAnalyzer` instances, load data, and convert to
`AnalysisData` containers.

The `AnalysisData` format supports swapping the **data representation**
(raw time-domain, ICA activations, wavelet amplitude/power, mean response, …)
while keeping all downstream analysis and visualisation code **identical**.

In [ ]:
# process_and_save_data = True
process_and_save_data = False


def make_analyzer(music_type):
    """Create, load and normalise an analyzer for one music type."""
    a = EEGSummarizedAnalyzer(
        experiment_name=ExperimentNames.PSILO_MUSIC,
        coordinate_system=CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS,
        music_types=[music_type],
        conditions=[ConditionVariants.PLACEBO],
        exclusion_categories=[ExclusionCategories.BAD_MUSIC],
    )
    if process_and_save_data:
        a.load_and_prepare_data(resample_freq=250.0, n_jobs=-1)
        print(f"[{music_type.value}] data shape: {a.data.shape}")
        a.save_data()
    else:
        a.load_data(
            info_filename=a.filtered_df[SingleDataMetadata.FILENAME].iloc[0],
        )
        print(f"[{music_type.value}] Loaded data shape: {a.data.shape}")
    a.normalize()
    return a


analyzer_classical = make_analyzer(MusicTypeVariants.CLASSICAL)
analyzer_psytrance = make_analyzer(MusicTypeVariants.PSYTRANCE)

# Ordered dict for easy iteration in all subsequent cells
analyzers = {
    "Classical": analyzer_classical,
    "Psytrance": analyzer_psytrance,
}

In [ ]:
# ============================================================================
# REPRESENTATION SWITCH
# ============================================================================
# Change the active block below to run ALL subsequent analyses on a
# different data representation.  Everything downstream uses `datasets`.
# ============================================================================

# --- Option 1: Raw time-domain EEG (default) ---
datasets = {label: a.to_analysis_data(label=label) for label, a in analyzers.items()}

# --- Option 2: Hilbert amplitude envelope (broadband) ---
# datasets = {
#     label: to_analytic_amplitude(a.to_analysis_data(label=label))
#     for label, a in analyzers.items()
# }

# --- Option 3: Wavelet amplitude (e.g. alpha band 8-13 Hz) ---
# freqs = np.arange(8, 14, 0.5)
# datasets = {
#     label: to_wavelet_amplitude(a.to_analysis_data(label=label), freqs=freqs)
#     for label, a in analyzers.items()
# }

# Summary
for label, ad in datasets.items():
    print(f"[{label}] {ad}")

## Data Overview

High-level summary of the loaded data array: number of subjects, channels,
time-points, recording duration and sampling frequency.

In [ ]:
# Data overview for all datasets
print_data_overview(datasets)

## Mean & Variance of the Signal (Global)

For each feature (electrode / IC component) the signal is first averaged
across all subjects at every time point.  Then the **temporal mean** and
**temporal variance** of that average signal are computed.

These per-feature summary statistics give an indication of the overall
signal level and its variability, complementing the ISC correlation
analysis.  Conditions are shown side-by-side.

In [ ]:
# Compute global mean & variance for all datasets
mean_var_results = {}
for label, ad in datasets.items():
    mean_f, var_f = compute_mean_variance(ad.data)
    mean_var_results[label] = (mean_f, var_f)
    print(
        f"[{label}]  mean range: [{mean_f.min():.4f}, {mean_f.max():.4f}]  "
        f"var range: [{var_f.min():.4f}, {var_f.max():.4f}]"
    )

In [ ]:
_first_ad = next(iter(datasets.values()))

plot_mean_variance_distribution(
    mean_var_results,
    feature_axis_label=f"Number of {_first_ad.feature_axis_label.lower()}s",
    save_path=SAVE_DIR / "mean_variance_distribution.png",
)

## Sliding-Window Mean & Variance (Time-Resolved)

Instead of collapsing the entire recording into a single mean/variance value,
a sliding window (default 5 s, step 2.5 s) is moved across time.  Inside
each window the signal is averaged across subjects, and the temporal mean
and variance within the window are computed per feature.

This reveals **when** during the recording the signal level or variability
changes, providing a time-resolved view analogous to the sliding-window ISC.

In [ ]:
# Compute sliding-window mean & variance for all datasets
sw_mv_results = {}
for label, ad in datasets.items():
    mean_tc, var_tc, sw_times = compute_sliding_window_mean_variance(
        ad.data, window_sec=WINDOW_SEC, step_sec=STEP_SEC, sfreq=ad.sfreq
    )
    sw_mv_results[label] = (mean_tc, var_tc, sw_times)
    print(
        f"[{label}]  mean_tc: {mean_tc.shape}  var_tc: {var_tc.shape}  "
        f"times: {sw_times.shape}"
    )

In [ ]:
_first_ad = next(iter(datasets.values()))

plot_sliding_window_mean_variance(
    sw_mv_results,
    feature_axis_label=f"{_first_ad.feature_axis_label} index",
    save_path=SAVE_DIR / "sliding_window_mean_variance.png",
)

## Band Mean & Variance (Global)

Same as the broadband analysis but each frequency band is band-pass filtered
first (delta 1–4 Hz, theta 4–8 Hz, alpha 8–13 Hz, beta 13–30 Hz, gamma
30–70 Hz).  This reveals which spectral components drive any amplitude
differences between conditions.

In [ ]:
# Compute per-band global mean & variance using the analyzer convenience method
band_mv_results = {}
for label, a in analyzers.items():
    band_mv_results[label] = a.compute_band_mean_variance()
    for band, (mf, vf) in band_mv_results[label].items():
        print(
            f"[{label}] {band:6s}  mean range: [{mf.min():.4f}, {mf.max():.4f}]  "
            f"var range: [{vf.min():.4f}, {vf.max():.4f}]"
        )

In [ ]:
_first_ad = next(iter(datasets.values()))

plot_band_mean_variance_distributions(
    band_mv_results,
    feature_axis_label=f"Number of {_first_ad.feature_axis_label.lower()}s",
    save_path=SAVE_DIR / "band_mean_variance_distribution.png",
)

## Band Sliding-Window Mean & Variance (Time-Resolved)

Time-resolved mean and variance for each frequency band.  Each panel shows
the feature-averaged trace (top), per-band variance trace (middle), and the
per-feature heatmap (bottom).

In [ ]:
# Compute per-band sliding-window mean & variance using the analyzer convenience method
band_sw_mv_results = {}
for label, a in analyzers.items():
    band_sw_mv_results[label] = a.compute_band_sliding_window_mean_variance(
        window_sec=WINDOW_SEC, step_sec=STEP_SEC
    )
    for band, (mtc, vtc, t) in band_sw_mv_results[label].items():
        print(
            f"[{label}] {band:6s}  mean_tc: {mtc.shape}  "
            f"var_tc: {vtc.shape}  times: {t.shape}"
        )

In [ ]:
_first_ad = next(iter(datasets.values()))

plot_band_sliding_window_mean_variance(
    band_sw_mv_results,
    feature_axis_label=f"{_first_ad.feature_axis_label} index",
    save_path=SAVE_DIR / "band_sliding_window_mean_variance.png",
)